In [ ]:
#LOAD DATA
import pandas as pd

df = pd.read_csv("training_data.csv")
if "Unnamed: 133" in df.columns:
    df = df.drop("Unnamed: 133", axis=1)

df.head()

,itching,skin_rash,nodal_skin_eruptions,continuous_sneezing,shivering,chills,joint_pain,stomach_pain,acidity,ulcers_on_tongue,...,blackheads,scurring,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,blister,red_sore_around_nose,yellow_crust_ooze,prognosis
0,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
1,0,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
2,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection
4,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Fungal infection


In [ ]:
import random
import pandas as pd

augmented_rows = []

for _, row in df.iterrows():
    symptoms = [col for col in df.columns[:-1] if row[col] == 1]

    # skip empty symptom rows (safety)
    if len(symptoms) == 0:
        continue

    for _ in range(10):  # increase samples
        new_row = {col: 0 for col in df.columns[:-1]}

        # 1. random subset of symptoms
        k = random.randint(1, len(symptoms))
        selected = random.sample(symptoms, k)

        # 2. add noise symptoms
        noise = random.sample(list(df.columns[:-1]), random.randint(0, 2))

        for s in selected + noise:
            new_row[s] = 1

        new_row["prognosis"] = row["prognosis"]
        augmented_rows.append(new_row)

# convert to dataframe
aug_df = pd.DataFrame(augmented_rows)

# combine original + augmented
df = pd.concat([df, aug_df], ignore_index=True)

print("New dataset shape:", df.shape)

New dataset shape: (54120, 133)


In [ ]:
#SPLIT FEATURES & LABEL
X = df.drop("prognosis", axis=1)
y = df["prognosis"]

In [ ]:
#ENCODE LABELS
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
#Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test_real)
lr_acc = accuracy_score(y_test_real_encoded, y_pred_lr)

results["Logistic Regression"] = lr_acc

In [ ]:
# MODEL 1 — RANDOM FOREST(ML)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [ ]:
#Load test data
test_df = pd.read_csv("test_data.csv")
if "Unnamed: 133" in test_df.columns:
    test_df = test_df.drop("Unnamed: 133", axis=1)
X_test_real = test_df.drop("prognosis", axis=1)
y_test_real = test_df["prognosis"]

# encode using SAME encoder
y_test_real_encoded = le.transform(y_test_real)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
import matplotlib.pyplot as plt

# Build model
input_dim = X_train.shape[1]
num_classes = len(set(y_encoded))

ann_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

# Compile
ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train
history = ann_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.7277 - loss: 1.1543 - val_accuracy: 0.8383 - val_loss: 0.5726
Epoch 2/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8463 - loss: 0.5372 - val_accuracy: 0.8524 - val_loss: 0.5109
Epoch 3/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8541 - loss: 0.4920 - val_accuracy: 0.8566 - val_loss: 0.4885
Epoch 4/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8603 - loss: 0.4669 - val_accuracy: 0.8595 - val_loss: 0.4821
Epoch 5/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8638 - loss: 0.4509 - val_accuracy: 0.8575 - val_loss: 0.4828
Epoch 6/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8663 - loss: 0.4402 - val_accuracy: 0.8589 - val_loss: 0.4786
Epoch 7/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8695 - loss: 0.4315 - val_accuracy: 0.8564 - val_loss: 0.4748
Epoch 8/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.8698 - loss: 0.4238 - 

In [ ]:
# Evaluate on real test data
loss, acc = ann_model.evaluate(X_test_real, y_test_real_encoded)
print("ANN Test Accuracy:", acc)

# Store result
results = {}
results["ANN"] = acc

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9762 - loss: 0.1052    
ANN Test Accuracy: 0.976190447807312


In [ ]:
#DNN (Stronger Deep Learning Model)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout

dnn_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_dnn = dnn_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.2
)

Epoch 1/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.7823 - loss: 0.8159 - val_accuracy: 0.8618 - val_loss: 0.4555
Epoch 2/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8515 - loss: 0.5121 - val_accuracy: 0.8682 - val_loss: 0.4337
Epoch 3/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.8596 - loss: 0.4768 - val_accuracy: 0.8666 - val_loss: 0.4383
Epoch 4/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8667 - loss: 0.4491 - val_accuracy: 0.8592 - val_loss: 0.4698
Epoch 5/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8702 - loss: 0.4345 - val_accuracy: 0.8682 - val_loss: 0.4447
Epoch 6/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8726 - loss: 0.4172 - val_accuracy: 0.8654 - val_loss: 0.4532
Epoch 7/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8791 - loss: 0.4012 - val_accuracy: 0.8643 - val_loss: 0.4624
Epoch 8/10
2165/2165 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - accuracy: 0.8806 - loss: 0.3881 - 

In [ ]:
# Evaluate
loss, dnn_acc = dnn_model.evaluate(X_test_real, y_test_real_encoded)
print("DNN Test Accuracy:", dnn_acc)
# Store
results["DNN"] = dnn_acc

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.9762 - loss: 0.0571    
DNN Test Accuracy: 0.976190447807312


In [ ]:
print(results)

{'ANN': 0.976190447807312, 'DNN': 0.976190447807312}


In [ ]:
#TOP 3 PREDICTION SYSTEM
import numpy as np

def predict_top3(model, input_data):
    # get probabilities
    probs = model.predict(input_data)[0]

    # get top 3 indices
    top3_idx = np.argsort(probs)[-3:][::-1]

    # map to disease names
    top3_diseases = le.inverse_transform(top3_idx)
    top3_probs = probs[top3_idx]

    # print nicely
    for i in range(3):
        print(f"{i+1}. {top3_diseases[i]} ({top3_probs[i]*100:.2f}%)")

In [ ]:
import numpy as np

def create_input(symptom_list):
    input_vector = np.zeros(len(X.columns))

    for symptom in symptom_list:
        if symptom in X.columns:
            idx = list(X.columns).index(symptom)
            input_vector[idx] = 1

    return input_vector.reshape(1, -1)

sample = create_input(['itching', 'skin_rash', 'fatigue'])

predict_top3(ann_model, sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1. Chicken pox (61.26%)
2. Jaundice (16.72%)
3. Fungal infection (7.98%)


In [ ]:
sample = create_input(['itching', 'skin_rash', 'fatigue'])
predict_top3(dnn_model, sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step
1. Fungal infection (51.13%)
2. Drug Reaction (35.33%)
3. Chicken pox (5.03%)


In [ ]:
import random
import numpy as np

def random_sample(n=3):
    symptoms = random.sample(list(X.columns), n)
    print("Selected Symptoms:", symptoms)

    input_vector = np.zeros(len(X.columns))

    for symptom in symptoms:
        idx = list(X.columns).index(symptom)
        input_vector[idx] = 1

    return input_vector.reshape(1, -1)

for i in range(5):
    sample = random_sample(3)
    print("\nTest Case", i+1)
    predict_top3(ann_model, sample)

Selected Symptoms: ['depression', 'drying_and_tingling_lips', 'muscle_pain']

Test Case 1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1. Fungal infection (8.57%)
2. hepatitis A (8.42%)
3. Impetigo (7.60%)
Selected Symptoms: ['sweating', 'redness_of_eyes', 'yellowing_of_eyes']

Test Case 2
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
1. Heart attack (18.23%)
2. Allergy (9.56%)
3. Hepatitis C (7.29%)
Selected Symptoms: ['extra_marital_contacts', 'knee_pain', 'runny_nose']

Test Case 3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1. AIDS (50.65%)
2. Osteoarthristis (10.20%)
3. Dimorphic hemmorhoids(piles) (5.97%)
Selected Symptoms: ['pain_during_bowel_movements', 'swelling_of_stomach', 'congestion']

Test Case 4
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
1. Dimorphic hemmorhoids(piles) (34.55%)
2. Alcoholic hepatitis (14.72%)
3. Urinary tract infection (4.03%)
Selected Symptoms: ['phlegm', 'blood_in_sputum', 'chest_pain']

Test Case 5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1. Tuberculosis (23.29%)
2. Hypertension

In [ ]:
prec_df = pd.read_csv("precautions.csv")
precautions = {}

for _, row in prec_df.iterrows():
    precautions[row["Disease"]] = [
        row["Precaution_1"],
        row["Precaution_2"],
        row["Precaution_3"]
    ]

In [ ]:
import numpy as np

def predict_with_precautions(model, input_data):
    probs = model.predict(input_data)[0]

    top3_idx = np.argsort(probs)[-3:][::-1]
    top3_diseases = le.inverse_transform(top3_idx)
    top3_probs = probs[top3_idx]

    for i in range(3):
        disease = top3_diseases[i]
        print(f"\n{i+1}. {disease} ({top3_probs[i]*100:.2f}%)")

        # show precautions
        if disease in precautions:
            print("Precautions:")
            for p in precautions[disease]:
                print("   -", p)
        else:
            print(" No precautions found")

In [ ]:
sample = create_input(['itching', 'skin_rash', 'fatigue'])
predict_with_precautions(ann_model, sample)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step

1. Fungal infection (56.33%)
Precautions:
   - Keep area dry
   - Use antifungal cream
   - Avoid sharing items

2. Drug Reaction (13.14%)
Precautions:
   - Stop medication
   - Consult doctor
   - Avoid allergens

3. Chicken pox (6.82%)
Precautions:
   - Avoid scratching
   - Maintain hygiene
   - Stay isolated
